# Working with Census Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/04-census-data.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Retrieve census block groups for any geographic area
- Query demographic variables using human-readable names
- Understand data quality issues (missing vs. suppressed data)
- Perform robust data aggregation handling None/negative values
- Create demographic profiles for accessibility analysis

## Prerequisites

- Completed [01-Getting Started](01-getting-started.ipynb)
- Completed [02-Isochrone Analysis](02-isochrone-analysis.ipynb)

## Setup

In [ ]:
# Install SocialMapper (pin versions for Colab compatibility)
!pip install -q "socialmapper[routing]" "pandas<3.0" "numpy<2.0"

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

In [ ]:
import os
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

# For production, set your Census API key:
# os.environ["CENSUS_API_KEY"] = "your-key-here"

import socialmapper
print(f"SocialMapper v{socialmapper.__version__}")

from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data
)
print("Ready!")

## Census Geography Overview

Census data is organized hierarchically:

```
Nation
  └── State
        └── County
              └── Census Tract
                    └── Block Group  ← SocialMapper uses this level
                          └── Block
```

**Block Groups** are the smallest geographic unit with detailed demographic data. They typically contain 600-3,000 people.

> **Tip:** Block groups are identified by a 12-digit GEOID: 2 (state) + 3 (county) + 6 (tract) + 1 (block group)

## Getting Census Block Groups

In [ ]:
# Get census blocks around a location
blocks = get_census_blocks(
    location=(45.5152, -122.6784),  # Portland
    radius_km=2
)

print(f"Found {len(blocks)} census block groups")

# Examine one block
block = blocks[0]
print(f"\nBlock structure:")
print(f"  GEOID: {block['geoid']}")
print(f"  State FIPS: {block['state_fips']}")
print(f"  County FIPS: {block['county_fips']}")
print(f"  Tract: {block['tract']}")
print(f"  Block Group: {block['block_group']}")
print(f"  Area: {block['area_sq_km']:.2f} km²")

## Getting Blocks from an Isochrone

In [ ]:
# Create an isochrone
isochrone = create_isochrone("Portland, OR", travel_time=15)

# Get census blocks within the isochrone
blocks = get_census_blocks(polygon=isochrone)

print(f"Census block groups in 15-min drive from Portland: {len(blocks)}")

# Total area
total_area = sum(b['area_sq_km'] for b in blocks)
print(f"Total area covered: {total_area:.2f} km²")

## Census Variables Reference

SocialMapper supports human-readable variable names that map to Census API codes.

| Variable Name | Description | Census Code |
|---------------|-------------|-------------|
| `population` | Total population | B01003_001E |
| `median_income` | Median household income | B19013_001E |
| `median_age` | Median age | B01002_001E |
| `total_households` | Total households | B11001_001E |
| `housing_units` | Total housing units | B25001_001E |
| `median_rent` | Median gross rent | B25064_001E |
| `median_home_value` | Median home value | B25077_001E |

> **Tip:** You can also use Census API codes directly (e.g., `B01003_001E`) for variables not in the mapping.

## Retrieving Census Data

In [ ]:
# Get GEOIDs from blocks
geoids = [b['geoid'] for b in blocks]

# Retrieve census data using human-readable names
result = get_census_data(
    location=geoids,
    variables=["population", "median_income"]
)

print(f"Query type: {result.location_type}")
print(f"Year: {result.query_info['year']}")
print(f"Block groups: {len(result.data)}")

# Show first few results
print("\nSample data:")
for geoid, data in list(result.data.items())[:3]:
    pop = data.get('population', 'N/A')
    income = data.get('median_income')
    income_str = f"${income:,}" if income and income > 0 else "N/A"
    print(f"  {geoid}: pop={pop}, income={income_str}")

## Three Ways to Query Census Data

In [ ]:
# Method 1: By polygon (isochrone)
isochrone = create_isochrone("Portland, OR", travel_time=15)
result1 = get_census_data(
    location=isochrone,
    variables=["population"]
)
print(f"Method 1 (polygon): {result1.location_type}, {len(result1.data)} blocks")

# Method 2: By GEOID list
sample_geoids = list(result1.data.keys())[:5]
result2 = get_census_data(
    location=sample_geoids,
    variables=["population"]
)
print(f"Method 2 (geoids): {result2.location_type}, {len(result2.data)} blocks")

# Method 3: By point (gets surrounding blocks)
result3 = get_census_data(
    location=(45.5152, -122.6784),  # Portland coordinates
    variables=["population"]
)
print(f"Method 3 (point): {result3.location_type}, {len(result3.data)} blocks")

## Understanding Data Quality

Census data can have **missing** or **suppressed** values:

| Value | Meaning | How to Handle |
|-------|---------|---------------|
| `None` | Data not available | Skip in calculations |
| `0` | Actual zero value | Include in calculations |
| Negative | Suppressed for privacy | Treat as missing |

> **Warning:** The Census Bureau suppresses data for small populations to protect privacy. Negative values indicate suppression, not actual negative numbers.

In [ ]:
# Check data quality in a census result
result = get_census_data(isochrone, variables=["median_income"])

# Count data quality categories
valid = 0
missing = 0
suppressed = 0
zero = 0

for geoid, data in result.data.items():
    income = data.get("median_income")
    if income is None:
        missing += 1
    elif income < 0:
        suppressed += 1  # Census suppresses data for privacy
    elif income == 0:
        zero += 1
    else:
        valid += 1

total = len(result.data)
print(f"Data Quality Report for median_income:")
print(f"  Total block groups: {total}")
print(f"  Valid data: {valid} ({valid/total*100:.1f}%)")
print(f"  Missing: {missing} ({missing/total*100:.1f}%)")
print(f"  Suppressed: {suppressed} ({suppressed/total*100:.1f}%)")
print(f"  Zero: {zero} ({zero/total*100:.1f}%)")

## Robust Data Aggregation

Always handle `None` and negative values when aggregating census data.

In [ ]:
def safe_sum(values, variable_name):
    """Safely sum values, handling None and negative (suppressed) values."""
    valid = [v for v in values if v is not None and v >= 0]
    return sum(valid), len(valid)

def safe_average(values, variable_name):
    """Safely average values, handling None and negative (suppressed) values."""
    valid = [v for v in values if v is not None and v > 0]
    if not valid:
        return None, 0
    return sum(valid) / len(valid), len(valid)

# Example: Aggregate population (summing)
result = get_census_data(isochrone, variables=["population", "median_income"])

populations = [data.get("population") for data in result.data.values()]
total_pop, pop_count = safe_sum(populations, "population")

incomes = [data.get("median_income") for data in result.data.values()]
avg_income, income_count = safe_average(incomes, "median_income")

print(f"Aggregation Results:")
print(f"  Total population: {total_pop:,} (from {pop_count} blocks)")
print(f"  Average median income: ${avg_income:,.0f} (from {income_count} blocks)" if avg_income else "  Average income: N/A")

In [ ]:
# Recommended pattern for robust aggregation
def aggregate_census_data(census_result, variable):
    """
    Robustly aggregate census data handling missing/suppressed values.
    
    For count variables (population, households): use sum
    For rate variables (income, age): use average
    """
    values = []
    missing = 0
    suppressed = 0
    
    for data in census_result.data.values():
        val = data.get(variable)
        if val is None:
            missing += 1
        elif val < 0:
            suppressed += 1
        else:
            values.append(val)
    
    # Count variables: sum
    if variable in ['population', 'total_households', 'housing_units']:
        result = sum(values)
    # Rate variables: average (excluding zeros for income)
    else:
        non_zero = [v for v in values if v > 0]
        result = sum(non_zero) / len(non_zero) if non_zero else None
    
    return {
        'value': result,
        'valid_count': len(values),
        'missing': missing,
        'suppressed': suppressed,
        'total': len(census_result.data)
    }

# Use the robust aggregation
pop_stats = aggregate_census_data(result, 'population')
income_stats = aggregate_census_data(result, 'median_income')

print("Population:")
print(f"  Total: {pop_stats['value']:,}")
print(f"  Data quality: {pop_stats['valid_count']}/{pop_stats['total']} blocks with data")

print("\nMedian Income:")
if income_stats['value']:
    print(f"  Average: ${income_stats['value']:,.0f}")
print(f"  Data quality: {income_stats['valid_count']}/{income_stats['total']} blocks with data")
if income_stats['suppressed'] > 0:
    print(f"  Warning: {income_stats['suppressed']} blocks have suppressed data")

## Population-Weighted Average

For more accurate income analysis, weight by population.

In [ ]:
# Get both population and income
result = get_census_data(
    isochrone,
    variables=["population", "median_income"]
)

# Calculate population-weighted average income
total_pop = 0
weighted_income = 0

for data in result.data.values():
    pop = data.get("population")
    income = data.get("median_income")
    
    # Only include if both values are valid
    if pop and pop > 0 and income and income > 0:
        total_pop += pop
        weighted_income += pop * income

if total_pop > 0:
    avg_income = weighted_income / total_pop
    print(f"Population-weighted average income: ${avg_income:,.0f}")
    print(f"Total population: {total_pop:,}")
else:
    print("Insufficient data for calculation")

## Demographic Profile Analysis

In [ ]:
def demographic_profile(location, travel_time=15):
    """Generate a demographic profile for an area with data quality reporting."""
    
    # Create area of interest
    isochrone = create_isochrone(location, travel_time=travel_time)
    
    # Get multiple variables
    result = get_census_data(
        isochrone,
        variables=["population", "median_income", "median_age", "total_households"]
    )
    
    # Aggregate data with quality tracking
    stats = {
        "population": {"total": 0, "valid": 0},
        "households": {"total": 0, "valid": 0},
        "incomes": [],
        "ages": []
    }
    
    for data in result.data.values():
        pop = data.get("population")
        if pop is not None and pop >= 0:
            stats["population"]["total"] += pop
            stats["population"]["valid"] += 1
            
        households = data.get("total_households")
        if households is not None and households >= 0:
            stats["households"]["total"] += households
            stats["households"]["valid"] += 1
            
        income = data.get("median_income")
        if income is not None and income > 0:  # Exclude suppressed
            stats["incomes"].append(income)
            
        age = data.get("median_age")
        if age is not None and age > 0:
            stats["ages"].append(age)
    
    # Print report
    print(f"\nDemographic Profile: {location}")
    print(f"({travel_time}-minute drive)")
    print("=" * 50)
    print(f"Total Population: {stats['population']['total']:,}")
    print(f"Total Households: {stats['households']['total']:,}")
    print(f"Block Groups: {len(result.data)}")
    
    if stats["incomes"]:
        print(f"\nIncome Range: ${min(stats['incomes']):,} - ${max(stats['incomes']):,}")
        print(f"Average Median Income: ${sum(stats['incomes'])//len(stats['incomes']):,}")
        print(f"  (based on {len(stats['incomes'])}/{len(result.data)} blocks with data)")
    else:
        print("\nIncome data: Not available")
    
    if stats["ages"]:
        print(f"\nAge Range: {min(stats['ages']):.1f} - {max(stats['ages']):.1f} years")
        print(f"Average Median Age: {sum(stats['ages'])/len(stats['ages']):.1f} years")
    
    return stats

# Generate profile
stats = demographic_profile("Portland, OR", travel_time=15)

## Income Inequality Analysis

In [ ]:
import statistics

def analyze_income_inequality(location):
    """Analyze income inequality in an area."""
    
    isochrone = create_isochrone(location, travel_time=20)
    result = get_census_data(isochrone, variables=["median_income", "population"])
    
    # Extract valid income data
    income_data = [
        {"income": d["median_income"], "pop": d["population"]}
        for d in result.data.values()
        if d.get("median_income") and d["median_income"] > 0
        and d.get("population") and d["population"] > 0
    ]
    
    if not income_data:
        print("No valid income data available")
        return
    
    incomes = [d["income"] for d in income_data]
    
    print(f"\nIncome Inequality Analysis: {location}")
    print("=" * 45)
    print(f"Block groups analyzed: {len(incomes)}")
    print(f"\nDistribution:")
    print(f"  Minimum: ${min(incomes):,}")
    print(f"  Maximum: ${max(incomes):,}")
    print(f"  Mean: ${statistics.mean(incomes):,.0f}")
    print(f"  Median: ${statistics.median(incomes):,.0f}")
    
    if len(incomes) > 1:
        print(f"  Std Dev: ${statistics.stdev(incomes):,.0f}")
    
    # Income ratio
    ratio = max(incomes) / min(incomes)
    print(f"\nInequality Indicators:")
    print(f"  Income ratio (max/min): {ratio:.1f}x")
    
    # Calculate percentiles
    sorted_incomes = sorted(incomes)
    n = len(sorted_incomes)
    if n >= 10:
        p10 = sorted_incomes[n // 10]
        p90 = sorted_incomes[9 * n // 10]
        print(f"  90/10 ratio: {p90/p10:.1f}x")

# Analyze
analyze_income_inequality("Portland, OR")

## Combining Census Data with Geography

In [ ]:
# Get blocks with geography
isochrone = create_isochrone("Portland, OR", travel_time=15)
blocks = get_census_blocks(polygon=isochrone)

# Get census data
geoids = [b['geoid'] for b in blocks]
result = get_census_data(geoids, variables=["population", "median_income"])

# Combine data with geography (robust pattern)
for block in blocks:
    data = result.data.get(block['geoid'], {})
    
    # Handle population
    pop = data.get('population')
    block['population'] = pop if pop is not None and pop >= 0 else 0
    
    # Handle income (might be suppressed)
    income = data.get('median_income')
    block['median_income'] = income if income is not None and income > 0 else None
    
    # Calculate density (avoid division by zero)
    area = block['area_sq_km'] if block['area_sq_km'] > 0 else 0.01
    block['density'] = block['population'] / area

# Show top 5 by population density
print("Top 5 Block Groups by Population Density:")
print("=" * 50)

top_blocks = sorted(blocks, key=lambda x: x['density'], reverse=True)[:5]
for b in top_blocks:
    income_str = f"${b['median_income']:,}" if b['median_income'] else "N/A"
    print(f"  {b['geoid']}: {b['density']:.0f} people/km²")
    print(f"    Population: {b['population']:,}, Income: {income_str}")

## Comparing Locations

In [ ]:
def compare_demographics(location1, location2, travel_time=15):
    """Compare demographics between two locations."""
    
    results = {}
    
    for loc in [location1, location2]:
        iso = create_isochrone(loc, travel_time=travel_time)
        census = get_census_data(
            iso,
            variables=["population", "median_income", "median_age"]
        )
        
        # Robust aggregation
        pop = sum(
            d.get("population", 0) or 0
            for d in census.data.values()
            if d.get("population") is not None and d.get("population") >= 0
        )
        incomes = [
            d["median_income"]
            for d in census.data.values()
            if d.get("median_income") and d["median_income"] > 0
        ]
        ages = [
            d["median_age"]
            for d in census.data.values()
            if d.get("median_age") and d["median_age"] > 0
        ]
        
        results[loc] = {
            "population": pop,
            "avg_income": sum(incomes) / len(incomes) if incomes else None,
            "avg_age": sum(ages) / len(ages) if ages else None,
            "blocks": len(census.data)
        }
    
    # Print comparison
    print(f"\nDemographic Comparison ({travel_time}-min drive)")
    print("=" * 60)
    print(f"{'Metric':<25} {location1:<15} {location2:<15}")
    print("-" * 60)
    
    r1, r2 = results[location1], results[location2]
    print(f"{'Population':<25} {r1['population']:>12,} {r2['population']:>12,}")
    
    inc1 = f"${r1['avg_income']:>10,.0f}" if r1['avg_income'] else "N/A"
    inc2 = f"${r2['avg_income']:>10,.0f}" if r2['avg_income'] else "N/A"
    print(f"{'Avg Median Income':<25} {inc1:>12} {inc2:>12}")
    
    age1 = f"{r1['avg_age']:>12.1f}" if r1['avg_age'] else "N/A"
    age2 = f"{r2['avg_age']:>12.1f}" if r2['avg_age'] else "N/A"
    print(f"{'Avg Median Age':<25} {age1:>12} {age2:>12}")
    
    print(f"{'Block Groups':<25} {r1['blocks']:>12} {r2['blocks']:>12}")
    
    return results

# Compare two cities
compare_demographics("Portland, OR", "Durham, NC")

## Troubleshooting

### Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| `MissingAPIKeyError` | No Census API key | Get free key from census.gov |
| Negative income values | Data suppression | Filter out values < 0 |
| `None` values | Data not available | Use default value or skip |
| Wrong totals | Including suppressed data | Use robust aggregation pattern |
| Empty results | Invalid GEOID format | Check GEOID is 12 digits |

### Getting a Census API Key

1. Go to: https://api.census.gov/data/key_signup.html
2. Fill out the form (free, no approval needed)
3. Check your email for the key
4. Set: `os.environ["CENSUS_API_KEY"] = "your-key"`

In [ ]:
from socialmapper import MissingAPIKeyError, DataError

# Example: Handle census errors gracefully
def safe_get_census_data(location, variables):
    """Get census data with error handling."""
    try:
        return get_census_data(location, variables)
    except MissingAPIKeyError as e:
        print(f"API Key Error: {e}")
        print("Get a free key at: https://api.census.gov/data/key_signup.html")
        return None
    except DataError as e:
        print(f"Data Error: {e}")
        return None

# Test the safe function
result = safe_get_census_data("Portland, OR", ["population"])
if result:
    print(f"Success! Retrieved data for {len(result.data)} blocks")

## Next Steps

Continue with:

- **[Mapping & Visualization](05-mapping-visualization.ipynb)** - Visualize census data on maps
- **[Complete Workflow](06-complete-workflow.ipynb)** - Full analysis combining all features
- **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Apply demographics to equity analysis